In [2]:
import pandas as pd
import numpy as np

clients = pd.read_csv("data/raw/clients.csv", parse_dates=["join_date"])
trades = pd.read_csv("data/raw/trades.csv", parse_dates=["date"])
fx_daily = pd.read_csv("data/raw/fx_daily.csv", parse_dates=["date"])

print(f"\nclients df = {clients.shape}")
clients.info()
print(f"\ntrades df = {trades.shape}")
trades.info()
print(f"\nfx_daily df = {fx_daily.shape}")
fx_daily.info()

print(f"\ntrades df\n{trades['profit_usd'].describe()}")
print(f"\nclient df by country\n{clients['country'].value_counts()}")
print(f"\ntrade df by pair\n{trades['pair'].value_counts()}")


clients df = (200, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   client_id     200 non-null    int64         
 1   country       200 non-null    object        
 2   account_type  200 non-null    object        
 3   join_date     200 non-null    datetime64[ns]
 4   deposit_usd   200 non-null    int64         
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 7.9+ KB

trades df = (3000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   trade_id    3000 non-null   int64         
 1   date        3000 non-null   datetime64[ns]
 2   client_id   3000 non-null   int64         
 3   pair        3000 non-null   object        
 4   side        3000 non-null   obje

In [3]:
print(f"\n{trades[trades['profit_usd'] > 0]}")
print(f"\n{trades[(trades['pair'] == 'XAUUSD') & (trades['lots'] >= 1.0)]}")
print(f"\n{clients[clients['country'].isin(['Thailand', 'Malaysia'])]}")
print(f"\n{trades.nlargest(10, 'profit_usd')}")
print(f"\n{fx_daily[(fx_daily['pair'] == 'EURUSD') & (fx_daily['date'].dt.year == 2026) & (fx_daily['date'].dt.month == 1)]}")


      trade_id       date  client_id    pair  side  lots  profit_usd
0         2905 2025-07-01         89  XAUUSD   buy  1.00       74.97
1         1713 2025-07-01         27  EURUSD   buy  2.00       69.01
2          730 2025-07-01         58  GBPUSD  sell  0.05        0.17
3         2438 2025-07-01        197  EURUSD  sell  1.00      241.01
4         2623 2025-07-01          9  EURUSD   buy  0.10        1.18
...        ...        ...        ...     ...   ...   ...         ...
2992      2467 2026-06-30        120  XAUUSD   buy  0.50       97.86
2993       739 2026-06-30         40  XAUUSD   buy  0.10       24.30
2995      1903 2026-06-30        102  EURUSD   buy  0.01        1.05
2996      1784 2026-06-30        100  USDJPY  sell  0.05        0.78
2997        31 2026-06-30        144  GBPUSD   buy  0.01        0.95

[1734 rows x 7 columns]

      trade_id       date  client_id    pair  side  lots  profit_usd
0         2905 2025-07-01         89  XAUUSD   buy   1.0       74.97
14     

In [4]:
trades.groupby("pair")["profit_usd"].sum()

pair
EURUSD    5148.46
GBPUSD    4741.21
USDJPY    3867.33
XAUUSD    3504.20
Name: profit_usd, dtype: float64

In [5]:
trades.groupby("pair").agg(
    total_profit = ("profit_usd", "sum"),
    avg_profit = ("profit_usd", "mean"),
    total_trades = ("profit_usd", "count"),
    total_lots = ("lots", "sum")
).sort_values(["total_lots", "total_trades"], ascending=[False, True])

,total_profit,avg_profit,total_trades,total_lots
pair,,,,
XAUUSD,3504.20,4.545006,771,252.96
USDJPY,3867.33,5.371292,720,237.32
EURUSD,5148.46,6.801136,757,232.64
GBPUSD,4741.21,6.304801,752,226.86


In [6]:
trades.groupby(["pair", "side"])["profit_usd"].mean()

pair    side
EURUSD  buy     7.121931
        sell    6.481187
GBPUSD  buy     4.700273
        sell    7.826192
USDJPY  buy     1.394448
        sell    8.831662
XAUUSD  buy     3.477220
        sell    5.757729
Name: profit_usd, dtype: float64

In [7]:
trades.pivot_table(index="pair", columns="side", values="profit_usd", aggfunc="mean")

side,buy,sell
pair,,
EURUSD,7.121931,6.481187
GBPUSD,4.700273,7.826192
USDJPY,1.394448,8.831662
XAUUSD,3.477220,5.757729


In [13]:
trades["pair_avg"] = trades.groupby("pair")["profit_usd"].transform("mean")
print(trades)

      trade_id       date  client_id    pair  side  lots  profit_usd  pair_avg
0         2905 2025-07-01         89  XAUUSD   buy  1.00       74.97  4.545006
1         1713 2025-07-01         27  EURUSD   buy  2.00       69.01  6.801136
2          730 2025-07-01         58  GBPUSD  sell  0.05        0.17  6.304801
3         2438 2025-07-01        197  EURUSD  sell  1.00      241.01  6.801136
4         2623 2025-07-01          9  EURUSD   buy  0.10        1.18  6.801136
...        ...        ...        ...     ...   ...   ...         ...       ...
2995      1903 2026-06-30        102  EURUSD   buy  0.01        1.05  6.801136
2996      1784 2026-06-30        100  USDJPY  sell  0.05        0.78  5.371292
2997        31 2026-06-30        144  GBPUSD   buy  0.01        0.95  6.304801
2998      1282 2026-06-30         56  USDJPY  sell  0.10       -0.62  5.371292
2999       881 2026-06-30        120  XAUUSD  sell  1.00     -130.19  4.545006

[3000 rows x 8 columns]


In [17]:
trades["vs_avg"] = trades["profit_usd"] - trades["pair_avg"]
print(trades)

      trade_id       date  client_id    pair  side  lots  profit_usd  \
0         2905 2025-07-01         89  XAUUSD   buy  1.00       74.97   
1         1713 2025-07-01         27  EURUSD   buy  2.00       69.01   
2          730 2025-07-01         58  GBPUSD  sell  0.05        0.17   
3         2438 2025-07-01        197  EURUSD  sell  1.00      241.01   
4         2623 2025-07-01          9  EURUSD   buy  0.10        1.18   
...        ...        ...        ...     ...   ...   ...         ...   
2995      1903 2026-06-30        102  EURUSD   buy  0.01        1.05   
2996      1784 2026-06-30        100  USDJPY  sell  0.05        0.78   
2997        31 2026-06-30        144  GBPUSD   buy  0.01        0.95   
2998      1282 2026-06-30         56  USDJPY  sell  0.10       -0.62   
2999       881 2026-06-30        120  XAUUSD  sell  1.00     -130.19   

      pair_avg      vs_avg  
0     4.545006   70.424994  
1     6.801136   62.208864  
2     6.304801   -6.134801  
3     6.801136  234

In [20]:
for pair, group in trades.groupby("pair"):
    print(pair, len(group))

EURUSD 757
GBPUSD 752
USDJPY 720
XAUUSD 771
